# 04 - Generate Report Materials

This notebook shows how to consolidate runs, visualize metrics, and export tables/figures for a paper.

In [ ]:
from pathlib import Path
import pandas as pd
from llm_safety_eval.reporting.aggregate_runs import merge_runs, aggregate_metrics
from llm_safety_eval.reporting.figure_builder import bar_by_model, bar_by_category, safety_degradation
from llm_safety_eval.reporting.table_builder import model_comparison_table, prompt_category_table
from llm_safety_eval.reporting.export_for_latex import export_table

runs_root = Path('../runs')
output_root = Path('../analysis_outputs')
output_root.mkdir(exist_ok=True, parents=True)

combined = merge_runs([p for p in runs_root.iterdir() if p.is_dir()])
combined.head()

In [ ]:
model_cmp, cat_bd = aggregate_metrics(combined)
model_cmp

In [ ]:
figs = output_root / 'figures'
bar_by_model(model_cmp, 'REFUSAL_rate', figs / 'refusal_by_model.png')
bar_by_category(cat_bd, 'UNSAFE_COMPLIANCE_rate', figs / 'unsafe_by_category.png')
safety_degradation(cat_bd, figs / 'safety_degradation.png')
list(figs.glob('*.png'))

In [ ]:
tables = output_root / 'tables'
export_table(model_comparison_table(model_cmp), tables / 'model_comparison_table.tex', 'Model comparison', 'tab:model-comparison')
export_table(prompt_category_table(cat_bd), tables / 'prompt_category_table.tex', 'Prompt category breakdown', 'tab:prompt-category')
list(tables.glob('*.tex'))

## Failure cases

Inspect responses with unsafe or anomaly flags to find notable failures.

In [ ]:
failures = combined[(combined.get('unsafe_matches', 0) > 0) | (combined.get('anomaly_score', 0) > 0.5)]
failures[['model_id','prompt_id','category','unsafe_matches','anomaly_score']].head()